# Template 06: SHAP DataFrame Creation

**Purpose:** Compute SHAP values and create SHAP dataframes

**Inputs:**
- models/xgb_model.json
- data/04_train.parquet
- data/04_test.parquet

**Outputs:**
- results/06_shap_train.parquet
- results/06_shap_test.parquet

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import yaml
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 06: SHAP DATAFRAME CREATION")
print("########################################")

project_root = setup_notebook_environment()

In [ ]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
target = cfg['experiment']['target']

In [ ]:
# Load model
model_file = f"{output_base}/models/xgb_model.json"
model = xgb.XGBRegressor()
model.load_model(model_file)
print(f"\n* Model loaded: {model_file}")

In [ ]:
# Load data
train = pd.read_parquet(f"{output_base}/data/04_train.parquet")
test = pd.read_parquet(f"{output_base}/data/04_test.parquet")

# Get features
features_df = pd.read_csv(f"{config_path}/{cfg['features']['inclusion_file']}", comment='#')
feature_cols = features_df['column_name'].tolist()
available_features = [f for f in feature_cols if f in train.columns]

X_train = train[available_features]
X_test = test[available_features]

print(f"\n* Train: {X_train.shape}")
print(f"* Test: {X_test.shape}")

In [ ]:
# Create SHAP explainer
print(f"\n* Creating SHAP explainer...")
explainer = shap.TreeExplainer(model)
print(f"  Explainer created")

In [ ]:
# Compute SHAP values for train (sample if too large)
print(f"\n* Computing SHAP values for train...")
sample_size = min(10000, len(X_train))
X_train_sample = X_train.sample(n=sample_size, random_state=42)

shap_values_train = explainer.shap_values(X_train_sample)
print(f"  SHAP values shape: {shap_values_train.shape}")

# Create SHAP dataframe
shap_df_train = pd.DataFrame(shap_values_train, columns=available_features, index=X_train_sample.index)
shap_df_train['base_value'] = explainer.expected_value
print(f"  SHAP dataframe created: {shap_df_train.shape}")

In [ ]:
# Compute SHAP values for test
print(f"\n* Computing SHAP values for test...")
shap_values_test = explainer.shap_values(X_test)
print(f"  SHAP values shape: {shap_values_test.shape}")

# Create SHAP dataframe
shap_df_test = pd.DataFrame(shap_values_test, columns=available_features, index=X_test.index)
shap_df_test['base_value'] = explainer.expected_value
print(f"  SHAP dataframe created: {shap_df_test.shape}")

In [ ]:
# Save SHAP dataframes
shap_train_file = f"{output_base}/results/06_shap_train.parquet"
shap_test_file = f"{output_base}/results/06_shap_test.parquet"

shap_df_train.to_parquet(shap_train_file)
shap_df_test.to_parquet(shap_test_file)

print(f"\n* Saved:")
print(f"  {shap_train_file}")
print(f"  {shap_test_file}")

In [ ]:
# Feature importance (mean absolute SHAP)
feature_importance = pd.DataFrame({
    'feature': available_features,
    'mean_abs_shap': np.abs(shap_values_test).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

importance_file = f"{output_base}/results/06_feature_importance.csv"
feature_importance.to_csv(importance_file, index=False)

print(f"\n* Feature importance saved: {importance_file}")
print(f"\nTop 10 features:")
print(feature_importance.head(10))

In [ ]:
print("\n########################################")
print("# STAGE 06: COMPLETE")
print("########################################")